In [ ]:
!pip install -U spacy

In [ ]:
import spacy
from spacy import displacy
from collections import Counter
import pandas as pd
pd.options.display.max_rows = 600
pd.options.display.max_colwidth = 400

In [ ]:
!python -m spacy download ru_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 87.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import ru_core_news_md
nlp = ru_core_news_md.load()

In [ ]:
filepath = 'text.txt'
text = open(filepath, encoding='utf-8').read()
document = nlp(text)

In [ ]:
document.ents

(Володя,
 Петербурге,
 Москву,
 Москву,
 Малой Никитской,
 Сыромятниковым,
 Москвы,
 Володя,
 Марфа Коренева,
 Марта,
 Кореневой,
 Москвы,
 Марта,
 Володи Марта,
 Мартой,
 Марте,
 Кореневой,
 Володя,
 Петербурге,
 Москву,
 Листьеву,
 Марту,
 Кореневу,
 Марты,
 Марты,
 Володя,
 Малой Никитской,
 Марте,
 Марфа Коренева,
 Володя,
 Марте,
 Маремьянова,
 Марфу,
 Володя,
 Марту,
 Метербурге,
 Москву,
 Марту,
 Марта,
 Володю)

In [ ]:
for named_entity in document.ents:
    print(named_entity, named_entity.label_)

Володя PER
Петербурге LOC
Москву LOC
Москву LOC
Малой Никитской LOC
Сыромятниковым PER
Москвы LOC
Володя PER
Марфа Коренева PER
Марта PER
Кореневой PER
Москвы LOC
Марта PER
Володи Марта PER
Мартой PER
Марте PER
Кореневой PER
Володя PER
Петербурге LOC
Москву LOC
Листьеву PER
Марту PER
Кореневу PER
Марты PER
Марты PER
Володя PER
Малой Никитской LOC
Марте PER
Марфа Коренева PER
Володя PER
Марте PER
Маремьянова PER
Марфу PER
Володя PER
Марту PER
Метербурге LOC
Москву LOC
Марту LOC
Марта PER
Володю PER


In [ ]:
for named_entity in document.ents:
    if named_entity.label_ == "PER":
        print(named_entity)

Володя
Сыромятниковым
Володя
Марфа Коренева
Марта
Кореневой
Марта
Володи Марта
Мартой
Марте
Кореневой
Володя
Листьеву
Марту
Кореневу
Марты
Марты
Володя
Марте
Марфа Коренева
Володя
Марте
Маремьянова
Марфу
Володя
Марту
Марта
Володю


In [ ]:
for named_entity in document.ents:
    if named_entity.label_ == "LOC":
        print(named_entity)

Петербурге
Москву
Москву
Малой Никитской
Москвы
Москвы
Петербурге
Москву
Малой Никитской
Метербурге
Москву
Марту


In [ ]:
import math
number_of_chunks = 80

chunk_size = math.ceil(len(text) / number_of_chunks)

text_chunks = []

for number in range(0, len(text), chunk_size):
    text_chunk = text[number:number+chunk_size]
    text_chunks.append(text_chunk)

In [ ]:
chunked_documents = list(nlp.pipe(text_chunks))

In [ ]:
people = []

for document in chunked_documents:
    for named_entity in document.ents:
        if named_entity.label_ == "PER":
            people.append(named_entity.text)

people_tally = Counter(people)

df = pd.DataFrame(people_tally.most_common(), columns=['character', 'count'])
df

,character,count
0,Володя,5
1,Марта,2
2,Марте,2
3,Март,2
4,Ма,1
5,лой Никитской,1
6,осквы,1
7,Марфа Корене,1
8,Коре,1
9,Воло,1


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1DDJZWt18rMZXLd9rF_btb-0XBUYMzzEGfZt45lsv6hY/edit#gid=0


In [ ]:
places = []
for document in chunked_documents:
    for named_entity in document.ents:
        if named_entity.label_ == "LOC":
            places.append(named_entity.text)

places_tally = Counter(places)

df = pd.DataFrame(places_tally.most_common(), columns=['place', 'count'])
df

,place,count
0,Москву,4
1,Петербу,2
2,Марту,2
3,М,1
4,Марта,1
5,Москвы,1
6,Кореневу,1
7,Малой Никитской,1
8,Метербурге,1
9,Володю,1


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/13-SZYqxk15XdwmwEogIPZBNT1jshbmYrI4Nm-Z-F4K0/edit#gid=0


In [ ]:
from IPython.display import Markdown, display
import re

def get_ner_in_context(keyword, document, desired_ner_labels= False):

    if desired_ner_labels != False:
        desired_ner_labels = desired_ner_labels
    else:
        # all possible labels
        desired_ner_labels = list(nlp.get_pipe('ner').labels)

    #Iterate through all the sentences in the document and pull out the text of each sentence
    for sentence in document.sents:
        #process each sentence
        sentence_doc = nlp(sentence.text)
        for named_entity in sentence_doc.ents:
            #Check to see if the keyword is in the sentence (and ignore capitalization by making both lowercase)
            if keyword.lower() in named_entity.text.lower()  and named_entity.label_ in desired_ner_labels:
                #Use the regex library to replace linebreaks and to make the keyword bolded, again ignoring capitalization
                #sentence_text = sentence.text

                sentence_text = re.sub('\n', ' ', sentence.text)
                sentence_text = re.sub(f"{named_entity.text}", f"**{named_entity.text}**", sentence_text, flags=re.IGNORECASE)

                print('---')
                display(Markdown(f"**{named_entity.label_}**"))
                display(Markdown(sentence_text))

In [ ]:
for document in chunked_documents:
    get_ner_in_context('Марта', document)

---


**LOC**

сто **Марта** – вернулас

---


**PER**

о **Марта** покорила все

---


**PER**

**ди Марта** всегда была

In [ ]:
filepath = 'text1.txt'
text = open(filepath, encoding='utf-8').read()
document = nlp(text)

In [ ]:
for token in document:
    print(token.lemma_, token.pos_, token.dep_)

осенний ADJ amod
визит NOUN nsubj
на ADP case
никитский ADJ nmod

 SPACE dep
быть VERB nmod
холодный ADJ amod
, PUNCT punct
двадцатиградусный ADJ conj
ноябрьский ADJ amod
день NOUN ROOT
. PUNCT punct
сам ADJ amod
александр PROPN nsubj
чувствовать VERB ROOT
себя PRON obj
виноватый ADJ xcomp
и CCONJ cc
странный ADJ conj
, PUNCT punct
стоить VERB advcl
неодетый VERB obj
у ADP case
окно NOUN obl
свой DET det
старый ADJ amod
квартира NOUN nmod
на ADP case
никитский ADJ nmod
. PUNCT punct
за ADP case
окно NOUN obl
кружиться VERB ROOT
лёгкий ADJ amod
первый ADJ amod
снег NOUN nsubj
, PUNCT punct
делать VERB advcl
большой ADJ amod
город NOUN obj
тёмный ADJ xcomp
и CCONJ cc
чужой ADJ conj
. PUNCT punct


 SPACE dep
этот DET det
последний ADJ amod
визит NOUN nsubj
к ADP case
пожилой ADJ amod
экономка NOUN nmod
марфа PROPN appos
быть AUX cop
важный ADJ ROOT
и CCONJ cc
трудный ADJ conj
. PUNCT punct
он PRON nsubj
вспомнить VERB ROOT
её DET det
письмо NOUN obj
— PUNCT punct
нежный ADJ amod
, PUNCT 

In [ ]:
adjs = []
for token in document:
    if token.pos_ == 'ADJ':
        adjs.append(token.lemma_)

In [ ]:
adjs

['осенний',
 'никитский',
 'холодный',
 'двадцатиградусный',
 'ноябрьский',
 'сам',
 'виноватый',
 'странный',
 'старый',
 'никитский',
 'лёгкий',
 'первый',
 'большой',
 'тёмный',
 'чужой',
 'последний',
 'пожилой',
 'важный',
 'трудный',
 'нежный',
 'трусливый',
 'жёлтый',
 'знаменитый',
 'талантливый',
 'молодой',
 'похожий',
 'самого',
 'другой',
 'другими',
 'громкий',
 'звенящий',
 'молоденький',
 'хорошенький',
 'больший',
 'необычный',
 'радостный',
 'беспокойный',
 'ждёт',
 'тихим',
 'нежный',
 'маленький',
 'простой',
 'тёплый',
 'душистый',
 'розовый',
 'сама',
 'равнодушный',
 'спокойный',
 'тонкий',
 'прозрачный',
 'старенький',
 'крошечный',
 'зелёный',
 'голубой',
 'милый',
 'невольный',
 'чужой',
 'тяжёлый',
 'однообразный',
 'неестественный',
 'пошлый',
 'порядочный',
 'должный',
 'фальшивый',
 'бесполезный',
 'нужный',
 'самому',
 'старый',
 'красивый',
 'молодой',
 'умный',
 'похожий',
 'родной',
 'случайный',
 'необходимый',
 'главный',
 'равнодушный',
 'живой',
 'т

In [ ]:
adjs_tally = Counter(adjs)

In [ ]:
adjs_tally.most_common()

[('холодный', 3),
 ('старый', 3),
 ('никитский', 2),
 ('лёгкий', 2),
 ('чужой', 2),
 ('важный', 2),
 ('нежный', 2),
 ('молодой', 2),
 ('похожий', 2),
 ('больший', 2),
 ('радостный', 2),
 ('маленький', 2),
 ('тёплый', 2),
 ('равнодушный', 2),
 ('милый', 2),
 ('тяжёлый', 2),
 ('нашёл', 2),
 ('осенний', 1),
 ('двадцатиградусный', 1),
 ('ноябрьский', 1),
 ('сам', 1),
 ('виноватый', 1),
 ('странный', 1),
 ('первый', 1),
 ('большой', 1),
 ('тёмный', 1),
 ('последний', 1),
 ('пожилой', 1),
 ('трудный', 1),
 ('трусливый', 1),
 ('жёлтый', 1),
 ('знаменитый', 1),
 ('талантливый', 1),
 ('самого', 1),
 ('другой', 1),
 ('другими', 1),
 ('громкий', 1),
 ('звенящий', 1),
 ('молоденький', 1),
 ('хорошенький', 1),
 ('необычный', 1),
 ('беспокойный', 1),
 ('ждёт', 1),
 ('тихим', 1),
 ('простой', 1),
 ('душистый', 1),
 ('розовый', 1),
 ('сама', 1),
 ('спокойный', 1),
 ('тонкий', 1),
 ('прозрачный', 1),
 ('старенький', 1),
 ('крошечный', 1),
 ('зелёный', 1),
 ('голубой', 1),
 ('невольный', 1),
 ('однообра

In [ ]:
df = pd.DataFrame(adjs_tally.most_common(), columns=['adj', 'count'])
df[:100]

,adj,count
0,холодный,3
1,старый,3
2,никитский,2
3,лёгкий,2
4,чужой,2
5,важный,2
6,нежный,2
7,молодой,2
8,похожий,2
9,больший,2


In [ ]:
nouns = []
for token in document:
    if token.pos_ == 'NOUN':
        nouns.append(token.lemma_)

nouns_tally = Counter(nouns)

df = pd.DataFrame(nouns_tally.most_common(), columns=['noun', 'count'])
df[:100]

,noun,count
0,человек,4
1,визит,3
2,глаз,3
3,день,2
4,окно,2
5,девушка,2
6,взгляд,2
7,связь,2
8,квартира,1
9,снег,1


In [ ]:
verbs = [token.lemma_ for token in document if token.pos_ == 'VERB']

verbs_tally = Counter(verbs)

df = pd.DataFrame(verbs_tally.most_common(), columns=['verb', 'count'])
df[:100]

,verb,count
0,чувствовать,2
1,стоять,2
2,быть,1
3,стоить,1
4,неодетый,1
5,кружиться,1
6,делать,1
7,вспомнить,1
8,упоминать,1
9,прервать,1


In [ ]:
https://melaniewalsh.github.io/Intro-Cultural-Analytics/05-Text-Analysis/Multilingual/Russian/02-Named-Entity-Recognition-Russian.html